<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

Each stage has **two paths**:
- **🚧 Compute**: Run from scratch (for debugging a single episode)
- **☁️ Load**: Skip computation, load pre-computed results from GCS (for debugging later stages)

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `gs://dm-tapnet/mv-tap/droid/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `gs://dm-tapnet/mv-tap/droid/extrinsics/` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `gs://dm-tapnet/mv-tap/droid/tracks/` | Dense 3D point tracks |

---
## 0. Environment Setup

In [ ]:
# @title 0a. Clone repo & install dependencies
import os

REPO_DIR = "/content/droid"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/yangyi02/droid.git {REPO_DIR}
else:
    print(f"⏭️ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

%cd {REPO_DIR}
!bash setup.sh

In [ ]:
# @title 0b. Python imports & sys.path setup
import sys, os, json, random
import numpy as np
import torch
import mediapy as media

REPO_DIR = "/content/droid"
for p in [
    REPO_DIR,
    os.path.join(REPO_DIR, "third_party/s2m2/src"),
    os.path.join(REPO_DIR, "third_party/co-tracker"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)
os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# @title 🔄 Dev: Sync from GitHub + Hot Reload (run after pushing changes)
import importlib, subprocess

result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

import core.geometry, core.io, core.depth, core.physics, core.tracking, core.visualization
for mod in [core.geometry, core.io, core.depth, core.physics, core.tracking, core.visualization]:
    importlib.reload(mod)

import compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks
for mod in [compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks]:
    importlib.reload(mod)

# Clear cached trackers to force re-initialization with reloaded code
for _k in list(globals().keys()):
    if _k.startswith("_tracker_"):
        print(f"🧹 Clearing cached tracker: {_k}")
        del globals()[_k]

print("✅ All modules reloaded. Re-run cells below to test changes.")

In [ ]:
# @title 0c. Download DROID metadata (shared by all stages)

import urllib.request
import os
import json

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"
files = ["camera_serials.json", "episode_id_to_path.json", 
         "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)

for f in files:
    target_file = os.path.join(root_path, f)
    if not os.path.exists(target_file):
        print(f"Downloading: {f} ...")
        req = urllib.request.Request(f"{base_url}/{f}", headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req) as response, open(target_file, 'wb') as out_file:
                out_file.write(response.read())
        except Exception as e:
            print(f"❌ {f} Download failed: {e}")

def load_json(name):
    with open(os.path.join(root_path, name)) as f:
        return json.load(f)

serials_db = load_json(files[0])
id_to_path = load_json(files[1])
keep_ranges = load_json(files[2])
extrinsics_db = load_json(files[3])

with open("episodes_success.txt") as f:
    valid_ids = sorted([line.strip() for line in f if line.strip()])
print(f"✅ Metadata ready: {len(valid_ids)} episodes")

In [ ]:
# @title 0d. Select episode

# Option 1: Random
episode_id = random.choice(valid_ids)

# Option 2: Manual override (uncomment)
# episode_id = "ILIAD+5e938e3b+2023-07-20-11h-50m-51s"

print(f"🎯 Episode: {episode_id}")

In [ ]:
# @title 0e. Initialize scene_constants (lightweight, no SVO)
from compute_depth import init_episode

scene_constants = init_episode(
    episode_id,
    os.path.expanduser("~/droid_data/input/robotics/droid_raw/1.0.1"),
    id_to_path, serials_db, keep_ranges)
print(f"✅ scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Stage 1: Depth

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 1A. 🚧 COMPUTE depth from scratch (SVO decode + S2M2 + SAM)
# This cell installs ZED SDK + runs the full depth pipeline.
# Skip this entirely if using 1B (Load from GCS).

# --- Install ZED SDK (only runs once) ---
import shutil
if not shutil.which("ZED_Explorer"):
    !apt-get update -qq
    !apt-get install -y zstd
    sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
    !wget -q -O {sdk_installer} https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22
    !chmod +x {sdk_installer}
    !./{sdk_installer} silent runtime_only skip_tools
    !find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;
    print("✅ ZED SDK installed")
else:
    print("⏭️ ZED SDK already installed")
import pyzed.sl as sl

# --- Run depth pipeline ---
from compute_depth import (
    init_all_models, extract_svo_video,
    parse_robot_kinematics, align_temporal_streams, export_to_disk,
)
from core.depth import (
    compute_stereo_depth, build_universal_gripper_mask,
    distill_empirical_gripper_depth, inject_gripper_depth,
)

# Init models (only first time)
if 's2m2_model' not in dir():
    s2m2_model, run_stereo_matching, sam_predictor = init_all_models()

scene_constants = extract_svo_video(scene_constants)
scene_constants = parse_robot_kinematics(scene_constants)
scene_constants = align_temporal_streams(scene_constants)
scene_constants = compute_stereo_depth(
    scene_constants, s2m2_model, run_stereo_matching, device)

# Gripper refinement
wrist_serial = scene_constants["meta"].get("wrist_serial")
if wrist_serial and wrist_serial in scene_constants["camera"]:
    wrist_data = scene_constants["camera"][wrist_serial]
    if "raw_depth" in wrist_data:
        wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()
scene_constants = build_universal_gripper_mask(scene_constants, sam_predictor)
scene_constants = distill_empirical_gripper_depth(scene_constants)
scene_constants = inject_gripper_depth(scene_constants)

export_to_disk(scene_constants)
print("✅ Stage 1 COMPUTE complete")

In [ ]:
# @title 1B. ☁️ LOAD depth from GCS bucket (skip Stage 1 computation)
# Run this if depth was already computed by run_parallel.sh.

GCS_DEPTH = "gs://dm-tapnet/mv-tap/droid/depth"
local_cache = f"/content/droid_depth_cache/{episode_id}"
os.makedirs(local_cache, exist_ok=True)

# Download robot data
os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{local_cache}/' > /dev/null 2>&1")
robot_data = np.load(f"{local_cache}/robot.npz", allow_pickle=True)
for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
    if k in robot_data:
        scene_constants['robot'][k] = robot_data[k]
if 'valid_indices' in robot_data:
    scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
if 'wrist_serial' in robot_data:
    scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())
wrist_serial = scene_constants['meta'].get('wrist_serial')
print(f"  ✅ robot.npz loaded")

# Download per-camera data
base_files = ["video_left.mp4", "video_right.mp4",
              "video_left_raw.mp4", "video_right_raw.mp4",
              "raw_depth.npz", "calibration.npz"]

for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    cam_files = list(base_files)
    if cam == wrist_serial:
        cam_files.extend(["original_raw_depth.npz", "gripper_mask.npz", "gripper_depth.npz"])

    gcs_files = " ".join([f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in cam_files])
    os.system(f"gsutil -m cp {gcs_files} '{cam_dir}/' > /dev/null 2>&1")

    # Videos
    for mem_key, fname in [("video_rgb", "video_left.mp4"), ("video_right", "video_right.mp4"),
                           ("video_raw_rgb", "video_left_raw.mp4"), ("video_raw_right", "video_right_raw.mp4")]:
        vid_path = os.path.join(cam_dir, fname)
        if os.path.exists(vid_path):
            scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

    # Depth
    depth_path = os.path.join(cam_dir, "raw_depth.npz")
    if os.path.exists(depth_path):
        scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

    # Wrist extras
    for npz_key, mem_key, is_depth in [
        ("original_raw_depth.npz", "original_raw_depth", True),
        ("gripper_mask.npz", "sam_real_masks", False),
        ("gripper_depth.npz", "empirical_gripper_depth", True)]:
        p = os.path.join(cam_dir, npz_key)
        if os.path.exists(p):
            d = np.load(p)
            key = 'depth' if 'depth' in d else 'mask'
            val = d[key]
            if is_depth:
                val = val.astype(np.float32) / 1000.0
            scene_constants['camera'][cam][mem_key] = val

    # Calibration
    calib_path = os.path.join(cam_dir, "calibration.npz")
    if os.path.exists(calib_path):
        c = np.load(calib_path)
        scene_constants['camera'][cam]['K_mat'] = c['K_calib_left']
        scene_constants['camera'][cam]['baseline'] = float(c['baseline'])
        scene_constants['camera'][cam]['zed_calibration'] = {
            'calibrated': {'K': c['K_calib_left'], 'disto': c['disto_calib_left'],
                           'K_right': c['K_calib_right'], 'disto_right': c['disto_calib_right']},
            'raw': {'K': c['K_raw_left'], 'disto': c['disto_raw_left'],
                    'K_right': c['K_raw_right'], 'disto_right': c['disto_raw_right']},
        }
    print(f"  ✅ Camera {cam} loaded")

print(f"✅ Stage 1 LOADED from GCS")

In [ ]:
# @title 1. Visualize depth results
from core.visualization import inspect_dict_structure, render_multicam_disparity_video

inspect_dict_structure(scene_constants)

frames = render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

---
## 1.5 Per-View 2D Point Tracking

Run **CoTracker**, **TAPNext**, and **AllTracker** on all cameras.
Populates `tracking_results` for downstream track-reprojection metrics in Stage 2.

In [ ]:
# @title 1.5a. Run 2D tracking (CoTracker + TAPNext + AllTracker)
import importlib, compute_2d_tracks
importlib.reload(compute_2d_tracks)
from compute_2d_tracks import init_tracker, run_2d_tracking

GRID_SIZE = 15  # @param {type:"integer"}
METHODS = ("cotracker", "tapnext", "alltracker")  # @param

tracking_results = {}
for method in METHODS:
    _cache_key = f"_tracker_{method}"
    if _cache_key not in dir():
        try:
            globals()[_cache_key] = init_tracker(method, device)
        except Exception as e:
            print(f"  ⚠️ {method}: init failed — {e}")
            continue
    scene_constants = run_2d_tracking(
        globals()[_cache_key], scene_constants, device, grid_size=GRID_SIZE)
    tracking_results[method] = {
        cam_id: {
            'tracks_2d': scene_constants['camera'][cam_id]['tracks_2d'].copy(),
            'vis_2d': scene_constants['camera'][cam_id]['vis_2d'].copy(),
        }
        for cam_id in scene_constants['camera']
        if 'tracks_2d' in scene_constants['camera'][cam_id]
    }

print(f"✅ tracking_results ready: {list(tracking_results.keys())}")

In [ ]:
# @title 1.5b. Tracking video (select method)
import importlib, core.visualization
importlib.reload(core.visualization)
from core.visualization import render_2d_tracking_video

VIS_METHOD = "cotracker"  # @param ["cotracker", "tapnext", "alltracker"]

if VIS_METHOD not in tracking_results:
    print(f"⚠️ Method '{VIS_METHOD}' not in tracking_results. "
          f"Available: {list(tracking_results.keys())}")
else:
    all_frames = []
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]
        if cam_id not in tracking_results[VIS_METHOD]:
            continue
        tracks = tracking_results[VIS_METHOD][cam_id]['tracks_2d']
        vis = tracking_results[VIS_METHOD][cam_id]['vis_2d']
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        all_frames.append(np.array(frames))

    if all_frames:
        combined = np.concatenate(all_frames, axis=2)
        media.show_video(combined, fps=10,
                         title=f"2D Tracks [{VIS_METHOD}] — All Cameras")

---
## 2. Stage 2: Extrinsics

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 2A. 🚧 COMPUTE extrinsics from scratch (VGGT + robot alignment)

from compute_extrinsics import (
    init_extrinsics,
    run_stage2_alignment, run_global_joint_alignment,
    evaluate_extrinsics, print_metrics, prepare_track_anchors,
    export_extrinsics,
)
from core.physics import PyBulletRenderer

# Init renderers (only first time)
if 'tensor_renderer' not in dir():
    from core.physics import TensorRobotRenderer
    tensor_renderer = TensorRobotRenderer(device=device)
if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

def _eval_and_print(scene_state, stage_name):
    """Evaluate metrics + dual-tracker track reproj in one shot."""
    base = evaluate_extrinsics(scene_constants, scene_state, device,
                               pb_renderer=pb_renderer)
    print_metrics(base, stage_name)
    for method, tr in tracking_results.items():
        try:
            for cid, d in tr.items():
                scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
                scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
            anchors = prepare_track_anchors(
                scene_constants, scene_state, pb_renderer, device)
            m = evaluate_extrinsics(scene_constants, scene_state, device,
                                    pb_renderer=pb_renderer,
                                    track_anchors=anchors)
            wbg = m.get('track_reproj_wrist_bg_mean_px', float('nan'))
            wbg_med = m.get('track_reproj_wrist_bg_median_px', float('nan'))
            rob = m.get('track_reproj_static_robot_mean_px', float('nan'))
            rob_med = m.get('track_reproj_static_robot_median_px', float('nan'))
            parts = []
            if not np.isnan(wbg):
                parts.append(f"wrist_bg={wbg:.2f}/{wbg_med:.2f}")
            if not np.isnan(rob):
                parts.append(f"robot_fk={rob:.2f}/{rob_med:.2f}")
            print(f"  🎯 Track ({method:>10s}): {' | '.join(parts)} px (mean/median)")
        except Exception as e:
            print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

# Stage 0+1: Dataset extrinsics → VGGT fallback
_vggt = globals().get('_vggt_models', None)
scene_state, _vggt_models = init_extrinsics(
    scene_constants, extrinsics_db, device, vggt_models=_vggt)
_eval_and_print(scene_state, "Stage 0+1 (Init)")

# Stage 2: Unified camera-robot alignment
scene_state = run_stage2_alignment(
    scene_constants, tensor_renderer, scene_state)
_eval_and_print(scene_state, "Stage 2 (Per-Camera)")

# Stage 3: Global joint optimization
scene_state = run_global_joint_alignment(
    scene_constants, scene_state, tensor_renderer)
_eval_and_print(scene_state, "Stage 3 (Global Joint)")

export_extrinsics(scene_constants, scene_state)
print("✅ Stage 2 COMPUTE complete")

In [ ]:
# @title 2B. ☁️ LOAD extrinsics from GCS bucket (skip extrinsics computation)

GCS_EXT = "gs://dm-tapnet/mv-tap/droid/extrinsics"
local_ext_cache = f"/content/droid_extrinsics_cache/{episode_id}"
os.makedirs(local_ext_cache, exist_ok=True)

scene_state = {}
for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_ext_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    gcs_path = f"{GCS_EXT}/{episode_id}/{cam}/extrinsics.json"
    local_path = os.path.join(cam_dir, "extrinsics.json")
    os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

    if os.path.exists(local_path):
        with open(local_path) as f:
            ext_data = json.load(f)
        scene_state[cam] = {
            'base_extrinsic': np.array(ext_data['base_extrinsic'], dtype=np.float32),
            'extrinsics': np.array(ext_data['extrinsics'], dtype=np.float32),
            'is_wrist': ext_data.get('is_wrist', False),
        }
        flag = "🦿" if scene_state[cam]['is_wrist'] else "🎥"
        print(f"  ✅ [{cam}] {flag} Shape: {scene_state[cam]['extrinsics'].shape}")
    else:
        print(f"  ⚠️ [{cam}] missing")

# --- Evaluate loaded extrinsics ---
import importlib, compute_extrinsics
importlib.reload(compute_extrinsics)
from compute_extrinsics import evaluate_extrinsics, print_metrics, prepare_track_anchors
from core.physics import PyBulletRenderer, TensorRobotRenderer

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()
if 'tensor_renderer' not in dir():
    tensor_renderer = TensorRobotRenderer(device=device)

# --- Compare PyBullet vs yourdfpy metrics ---
import time

t0 = time.time()
base_yourdfpy = evaluate_extrinsics(scene_constants, scene_state, device,
                                     tensor_renderer=tensor_renderer)
t_yourdfpy = time.time() - t0

t0 = time.time()
base_pybullet = evaluate_extrinsics(scene_constants, scene_state, device,
                                     pb_renderer=pb_renderer)
t_pybullet = time.time() - t0

print(f"⏱️  yourdfpy: {t_yourdfpy:.1f}s  |  PyBullet: {t_pybullet:.1f}s  |  speedup: {t_pybullet/max(t_yourdfpy, 0.01):.1f}×\n")

# Side-by-side comparison
for key in sorted(set(base_yourdfpy) | set(base_pybullet)):
    v1 = base_yourdfpy.get(key, float('nan'))
    v2 = base_pybullet.get(key, float('nan'))
    if isinstance(v1, float) and isinstance(v2, float):
        delta = abs(v1 - v2)
        flag = "⚠️" if delta > 0.01 else "✅"
        print(f"  {flag} {key:40s}  yourdfpy={v1:.4f}  pybullet={v2:.4f}  Δ={delta:.4f}")

print_metrics(base_yourdfpy, "Final Extrinsics — yourdfpy (fast)")
print_metrics(base_pybullet, "Final Extrinsics — PyBullet (ref)")

# --- Track reprojection (uses pb_renderer for prepare_track_anchors) ---
for method, tr in tracking_results.items():
    try:
        for cid, d in tr.items():
            scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
            scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
        anchors = prepare_track_anchors(
            scene_constants, scene_state, pb_renderer, device)
        m = evaluate_extrinsics(scene_constants, scene_state, device,
                                tensor_renderer=tensor_renderer,
                                track_anchors=anchors)
        wbg = m.get('track_reproj_wrist_bg_mean_px', float('nan'))
        wbg_med = m.get('track_reproj_wrist_bg_median_px', float('nan'))
        rob = m.get('track_reproj_static_robot_mean_px', float('nan'))
        rob_med = m.get('track_reproj_static_robot_median_px', float('nan'))
        parts = []
        if not np.isnan(wbg):
            parts.append(f"wrist_bg={wbg:.2f}/{wbg_med:.2f}")
        if not np.isnan(rob):
            parts.append(f"robot_fk={rob:.2f}/{rob_med:.2f}")
        print(f"  🎯 Track ({method:>10s}): {' | '.join(parts)} px (mean/median)")
    except Exception as e:
        print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

print("✅ Extrinsics LOADED from GCS")

In [ ]:
# @title 2a. Camera axes overlay
from core.visualization import render_cross_camera_axes

try:
    axes_frames = render_cross_camera_axes(scene_constants, scene_state, max_frames=30)
    if axes_frames:
        media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")
except Exception as e:
    print(f"Axes visualization skipped: {e}")

In [ ]:
# @title 2b. Robot segmentation video
from core.visualization import render_segmentation_video
from core.physics import PyBulletRenderer

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

try:
    seg_frames = render_segmentation_video(scene_constants, scene_state, pb_renderer, max_frames=30)
    if seg_frames:
        media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")
except Exception as e:
    print(f"Segmentation visualization skipped: {e}")

In [ ]:
# @title 2c. Fused 3D point cloud
from core.visualization import render_fused_point_cloud

try:
    render_fused_point_cloud(scene_constants, scene_state, frame_idx=0, height=600, width=1000)
except Exception as e:
    print(f"Fused point cloud skipped: {e}")

In [ ]:
# @title 2d. 4D cinematic orbit
from core.visualization import render_cinematic_4d_orbit

try:
    orbit_frames = render_cinematic_4d_orbit(scene_constants, scene_state, max_frames=30)
    media.show_video(orbit_frames, fps=10, title="4D Orbit")
except Exception as e:
    print(f"4D Orbit skipped: {e}")

In [ ]:
# @title 🔍 Visualize Extrinsics Metrics (all items with visual inputs)
# For each metric, first show WHAT goes into the computation (visual inputs),
# then show the resulting error curve.

import importlib, copy, torch, numpy as np, matplotlib.pyplot as plt
import cv2
import compute_extrinsics; importlib.reload(compute_extrinsics)
import compute_2d_tracks; importlib.reload(compute_2d_tracks)
import core.visualization; importlib.reload(core.visualization)
from compute_extrinsics import (prepare_track_anchors, compute_track_reproj_loss,
                                 evaluate_extrinsics, print_metrics,
                                 get_cam_points_local_t, batched_chamfer_distance)
from compute_2d_tracks import init_tracker, run_2d_tracking
from core.physics import PyBulletRenderer
from core.tracking import URDFKinematicsTracker
from core.pybullet_extrinsics import (
    get_foreground_robot_points, get_foreground_gripper_points,
    compute_robot_loss_batched, compute_wrist_loss_batched,
)
from core.visualization import (
    render_fused_point_cloud, render_segmentation_video,
    render_2d_tracking_video, render_cross_camera_axes,
)
import mediapy as media

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

wrist_cam = scene_constants['meta']['wrist_serial']
ext_cams = [c for c in scene_constants['camera'].keys() if c != wrist_cam]
n_frames = len(scene_constants['robot']['joint_positions'])
T_ee_all = scene_constants['robot']['T_ee_base_all']
VIS_FRAME = n_frames // 2  # pick mid-episode frame for input visualizations

# ============================================================
# 1. ROBOT DEPTH LOSS — rendered robot depth vs observed depth
# ============================================================
print("=" * 70)
print("📊 1. Robot Depth Loss (PyBullet rendered vs sensor depth)")
print("    Input: For each camera, compare PyBullet-rendered robot depth")
print("           against the observed sensor depth on robot pixels.")
print("=" * 70)

# --- 1a. VISUAL INPUTS: show rendered depth, observed depth, and diff ---
all_cam_ids = list(ext_cams) + [wrist_cam]
cam_labels = ['cam1 (ext)', 'cam2 (ext)', 'wrist']

joints_vis = scene_constants['robot']['joint_positions'][VIS_FRAME]
gripper_vis = scene_constants['robot']['gripper_positions'][VIS_FRAME]
pb_renderer.update_robot_pose(joints_vis, gripper_state=gripper_vis)

fig, axes = plt.subplots(3, len(all_cam_ids), figsize=(6*len(all_cam_ids), 14))
for col, (cam_id, label) in enumerate(zip(all_cam_ids, cam_labels)):
    K_np = scene_constants['camera'][cam_id]['K_mat']
    ext_t = scene_state[cam_id]['extrinsics'][VIS_FRAME]
    d_obs = scene_constants['camera'][cam_id]['raw_depth'][VIS_FRAME].astype(np.float32)
    h_img, w_img = d_obs.shape
    d_render = pb_renderer.render_depth(ext_t, K_np, w_img, h_img)
    robot_mask = d_render > 0.01

    # Row 0: rendered robot depth
    im0 = axes[0, col].imshow(np.where(robot_mask, d_render, np.nan),
                               cmap='viridis', vmin=0.1, vmax=1.2)
    axes[0, col].set_title(f'{label}\nRendered Robot Depth', fontsize=11)
    axes[0, col].axis('off')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046)

    # Row 1: observed sensor depth (robot region only)
    im1 = axes[1, col].imshow(np.where(robot_mask, d_obs, np.nan),
                               cmap='viridis', vmin=0.1, vmax=1.2)
    axes[1, col].set_title(f'{label}\nObserved Sensor Depth\n(robot region)', fontsize=11)
    axes[1, col].axis('off')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046)

    # Row 2: |diff| on robot pixels
    diff = np.abs(d_render - d_obs)
    valid = robot_mask & (d_obs > 0.01) & (d_obs < 1.5)
    diff_masked = np.where(valid, diff, np.nan)
    im2 = axes[2, col].imshow(diff_masked, cmap='hot', vmin=0, vmax=0.1)
    mean_err = np.nanmean(diff_masked) if np.any(valid) else 0
    axes[2, col].set_title(f'{label}\n|Rendered - Observed|\nmean={mean_err:.4f}m', fontsize=11)
    axes[2, col].axis('off')
    plt.colorbar(im2, ax=axes[2, col], fraction=0.046)

plt.suptitle(f'Robot Depth Loss Inputs (Frame {VIS_FRAME})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- 1b. Also show robot mask overlay using visualization.py ---
print("  🤖 Robot segmentation overlay (from core.visualization):")
from core.visualization import render_multiview_mask_inspection
render_multiview_mask_inspection(scene_constants, scene_state, pb_renderer, frame_idx=VIS_FRAME)

# --- 1c. Per-frame error curve ---
fig, axes_curve = plt.subplots(1, 3, figsize=(18, 4))
for ax, cam_id, label in zip(axes_curve, all_cam_ids, cam_labels):
    is_wrist = (cam_id == wrist_cam)
    K_np = scene_constants['camera'][cam_id]['K_mat']
    per_frame_err = []
    for t in range(n_frames):
        joints = scene_constants['robot']['joint_positions'][t]
        gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(joints, gripper_state=gripper)
        ext_t = scene_state[cam_id]['extrinsics'][t]
        d_obs = scene_constants['camera'][cam_id]['raw_depth'][t].astype(np.float32)
        h_img, w_img = d_obs.shape
        d_render = pb_renderer.render_depth(ext_t, K_np, w_img, h_img)
        valid = (d_render > 0.01) & (d_render < 1.5) & (d_obs > 0.01) & (d_obs < 1.5)
        if valid.any():
            per_frame_err.append(np.abs(d_render[valid] - d_obs[valid]).mean())
        else:
            per_frame_err.append(np.nan)
    per_frame_err = np.array(per_frame_err)
    ax.plot(per_frame_err, linewidth=0.8)
    ax.axhline(y=np.nanmean(per_frame_err), color='r', linestyle='--',
               label=f'mean={np.nanmean(per_frame_err):.4f}m')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Mean |Δ depth| (m)')
    ax.set_title(f'{label} [{cam_id[:8]}]')
    ax.legend(fontsize=8)
plt.suptitle('Robot Depth Loss per Frame', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================================
# 2. CHAMFER DISTANCE — 3D point cloud alignment
# ============================================================
print("\n" + "=" * 70)
print("📊 2. Chamfer Distance (3D point cloud consistency)")
print("    Input: Unproject depth from each camera to 3D world points,")
print("           then compute nearest-neighbor distance between pairs.")
print("=" * 70)

# --- 2a. VISUAL INPUT: Fused point cloud showing per-camera tinting ---
print("  🌐 Fused 3D point cloud (per-camera tint) — from core.visualization:")
render_fused_point_cloud(scene_constants, scene_state, frame_idx=VIS_FRAME,
                         use_tint=True, max_render_points=100000)

# --- 2b. Show the 2000 sampled points used in Chamfer computation ---
cam1, cam2 = ext_cams[0], ext_cams[1]
fig, axes_2d = plt.subplots(1, 3, figsize=(18, 5))
for ax, cam_id, label in zip(axes_2d, [cam1, cam2, wrist_cam],
                              ['cam1 (ext)', 'cam2 (ext)', 'wrist']):
    img = scene_constants['camera'][cam_id]['video_rgb'][VIS_FRAME].copy()
    d = scene_constants['camera'][cam_id]['raw_depth'][VIS_FRAME].astype(np.float32)
    K_np = scene_constants['camera'][cam_id]['K_mat']
    valid = (d > 0) & (d < 1.5)
    vs, us = np.where(valid)
    if len(us) > 2000:
        idx = np.random.choice(len(us), 2000, replace=False)
        us_s, vs_s = us[idx], vs[idx]
    else:
        us_s, vs_s = us, vs
    ax.imshow(img)
    ax.scatter(us_s, vs_s, c=d[vs_s, us_s], cmap='viridis', s=1, alpha=0.7,
               vmin=0.1, vmax=1.2)
    ax.set_title(f'{label}: {len(us_s)} sampled points\n(colored by depth)', fontsize=11)
    ax.axis('off')
plt.suptitle(f'Chamfer Distance Inputs — Sampled Depth Points (Frame {VIS_FRAME})', fontsize=14)
plt.tight_layout()
plt.show()

# --- 2c. Per-frame Chamfer curve ---
T1 = torch.tensor(scene_state[cam1]['base_extrinsic'], dtype=torch.float32, device=device)
T2 = torch.tensor(scene_state[cam2]['base_extrinsic'], dtype=torch.float32, device=device)
Tw = torch.tensor(scene_state[wrist_cam]['base_extrinsic'], dtype=torch.float32, device=device)

chamfer_12, chamfer_1w, chamfer_2w = [], [], []
for t in range(n_frames):
    pc1 = get_cam_points_local_t(t, scene_constants['camera'][cam1], device)
    pc2 = get_cam_points_local_t(t, scene_constants['camera'][cam2], device)
    pcw = get_cam_points_local_t(t, scene_constants['camera'][wrist_cam], device)
    if pc1 is None or pc2 is None or pcw is None:
        chamfer_12.append(np.nan); chamfer_1w.append(np.nan); chamfer_2w.append(np.nan)
        continue
    Tee_t = torch.tensor(T_ee_all[t], dtype=torch.float32, device=device)
    w1 = (T1 @ pc1)[:3, :].T.unsqueeze(0)
    w2 = (T2 @ pc2)[:3, :].T.unsqueeze(0)
    ww = ((Tee_t @ Tw) @ pcw)[:3, :].T.unsqueeze(0)
    l12, _ = batched_chamfer_distance(w1, w2, device)
    l1w, _ = batched_chamfer_distance(w1, ww, device)
    l2w, _ = batched_chamfer_distance(w2, ww, device)
    chamfer_12.append(l12.item()); chamfer_1w.append(l1w.item()); chamfer_2w.append(l2w.item())

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(chamfer_12, label=f'cam1↔cam2 (mean={np.nanmean(chamfer_12):.4f})', linewidth=0.8)
ax.plot(chamfer_1w, label=f'cam1↔wrist (mean={np.nanmean(chamfer_1w):.4f})', linewidth=0.8)
ax.plot(chamfer_2w, label=f'cam2↔wrist (mean={np.nanmean(chamfer_2w):.4f})', linewidth=0.8)
ax.set_xlabel('Frame'); ax.set_ylabel('Chamfer Distance (m)')
ax.set_title('Chamfer Distance per Frame (lower = better alignment)')
ax.legend()
plt.tight_layout(); plt.show()

# ============================================================
# 3. TRACK WRIST BG — FK-reprojected background tracks on wrist cam
# ============================================================
print("\n" + "=" * 70)
print("📊 3. Track Wrist Background (FK reproj vs 2D tracker)")
print("    Input: On wrist camera, select background (non-gripper) tracks.")
print("           At t=0, lift them to 3D via depth. Assume they are static")
print("           in world frame. Reproject using FK at each frame t.")
print("           Compare FK-predicted 2D positions vs tracker-predicted 2D.")
print("=" * 70)

tracker_name = "cotracker"
if 'dbg_tracker' not in dir() or dbg_tracker.name.lower() != tracker_name:
    dbg_tracker = init_tracker(tracker_name, device)

sc_dbg = copy.deepcopy(scene_constants)
sc_dbg = run_2d_tracking(dbg_tracker, sc_dbg, device, grid_size=30)
anchors = prepare_track_anchors(sc_dbg, scene_state, pb_renderer, device)
T_ee_t = torch.tensor(T_ee_all, dtype=torch.float32, device=device)

# --- 3a. VISUAL INPUT: Show wrist camera with background vs gripper tracks ---
if wrist_cam in anchors and anchors[wrist_cam]['scheme'] == 'background':
    wrist_data = sc_dbg['camera'][wrist_cam]
    wrist_anchor = anchors[wrist_cam]
    tracks_all = wrist_data['tracks_2d']  # (T, N, 2)
    vis_all = wrist_data['vis_2d']

    # Get gripper mask to distinguish bg vs robot tracks
    sam_masks = wrist_data.get('sam_real_masks')
    consensus = sam_masks.any(axis=0) if sam_masks is not None else np.zeros_like(wrist_data['raw_depth'][0], dtype=bool)
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_d = cv2.dilate(consensus.astype(np.uint8), kernel, iterations=1) > 0

    u0 = tracks_all[0, :, 0]
    v0 = tracks_all[0, :, 1]
    u0i = np.clip(np.round(u0).astype(int), 0, wrist_data['raw_depth'][0].shape[1] - 1)
    v0i = np.clip(np.round(v0).astype(int), 0, wrist_data['raw_depth'][0].shape[0] - 1)
    on_robot = robot_mask_d[v0i, u0i]

    # Show t=0 frame with bg tracks (green) and robot tracks (red)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    img0 = wrist_data['video_rgb'][0].copy()

    # Left: gripper mask + track classification
    overlay = img0.copy()
    overlay[robot_mask_d] = [255, 100, 100]
    blended = cv2.addWeighted(img0, 0.6, overlay, 0.4, 0)
    axes[0].imshow(blended)
    bg_pts = ~on_robot & vis_all[0]
    robot_pts = on_robot & vis_all[0]
    axes[0].scatter(u0[bg_pts], v0[bg_pts], c='lime', s=8, label=f'BG ({bg_pts.sum()})', zorder=5)
    axes[0].scatter(u0[robot_pts], v0[robot_pts], c='red', s=8, label=f'Robot ({robot_pts.sum()})', zorder=5)
    axes[0].legend(fontsize=10)
    axes[0].set_title(f'Wrist t=0: Track Classification\n'
                      f'BG tracks used for metric, robot tracks excluded', fontsize=11)
    axes[0].axis('off')

    # Right: visualize the BG tracks as a video frame overlay using core.visualization
    bg_track_indices = np.where(~on_robot & vis_all[0])[0]
    if len(bg_track_indices) > 0:
        bg_tracks = tracks_all[:, bg_track_indices]
        bg_vis = vis_all[:, bg_track_indices]
        # Show a single frame with track trails
        frames_vis = render_2d_tracking_video(
            wrist_data['video_rgb'][:min(30, n_frames)],
            bg_tracks[:min(30, n_frames)],
            bg_vis[:min(30, n_frames)],
            linewidth=2, tracks_leave_trace=15)
        mid = min(20, len(frames_vis)-1)
        axes[1].imshow(frames_vis[mid])
        axes[1].set_title(f'Wrist BG Tracks Overlay (frame {mid})\n'
                          f'These points should be static in world frame', fontsize=11)
        axes[1].axis('off')
    plt.suptitle('Track Wrist BG — Input Visualization', fontsize=14)
    plt.tight_layout()
    plt.show()

    # --- 3b. Show FK-predicted vs tracker-predicted on a sample frame ---
    T_opt = torch.tensor(scene_state[wrist_cam]['base_extrinsic'],
                         dtype=torch.float32, device=device)
    K_t = torch.tensor(sc_dbg['camera'][wrist_cam]['K_mat'],
                       dtype=torch.float32, device=device)
    P_cam0 = wrist_anchor['P_cam0']
    targets = wrist_anchor['tracks_2d']
    vis_bg = wrist_anchor['vis']

    T_cam_to_world_0 = T_ee_t[0] @ T_opt
    P_world = T_cam_to_world_0 @ P_cam0.T
    T_cam_to_world_all = T_ee_t @ T_opt.unsqueeze(0)
    T_world_to_cam_all = torch.linalg.inv(T_cam_to_world_all)
    P_cam_all = T_world_to_cam_all @ P_world.unsqueeze(0)
    Z = P_cam_all[:, 2, :].clamp(min=1e-4)
    u_pred = K_t[0, 0] * P_cam_all[:, 0, :] / Z + K_t[0, 2]
    v_pred = K_t[1, 1] * P_cam_all[:, 1, :] / Z + K_t[1, 2]
    pred = torch.stack([u_pred, v_pred], dim=-1).cpu().numpy()
    tgt_np = targets.cpu().numpy()
    vis_np = vis_bg.cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, t_show in zip(axes, [0, VIS_FRAME, min(n_frames-1, n_frames-5)]):
        img = wrist_data['video_rgb'][t_show].copy()
        ax.imshow(img)
        v_mask = vis_np[t_show]
        ax.scatter(tgt_np[t_show, v_mask, 0], tgt_np[t_show, v_mask, 1],
                   c='lime', s=12, label='Tracker (target)', zorder=5, alpha=0.7)
        ax.scatter(pred[t_show, v_mask, 0], pred[t_show, v_mask, 1],
                   c='red', s=12, marker='x', label='FK-predicted', zorder=5, alpha=0.7)
        # Draw lines between pairs
        for i in np.where(v_mask)[0][:50]:
            ax.plot([tgt_np[t_show, i, 0], pred[t_show, i, 0]],
                    [tgt_np[t_show, i, 1], pred[t_show, i, 1]],
                    'y-', alpha=0.3, linewidth=0.5)
        err_t = np.abs(pred[t_show, v_mask] - tgt_np[t_show, v_mask]).sum(axis=-1).mean()
        ax.set_title(f'Frame {t_show} | mean err={err_t:.1f}px', fontsize=11)
        ax.legend(fontsize=8)
        ax.axis('off')
    plt.suptitle('Wrist BG: FK-Predicted (red ×) vs Tracker (green ●)', fontsize=14)
    plt.tight_layout()
    plt.show()

    # --- 3c. Error curves ---
    pixel_err = np.abs(pred - tgt_np).sum(axis=-1)
    eval_mask = wrist_anchor.get('eval_frame_mask', torch.ones(n_frames, dtype=torch.bool))
    eval_mask_np = eval_mask.cpu().numpy()
    per_frame = [pixel_err[t, vis_np[t]].mean() if vis_np[t].any() and eval_mask_np[t] else np.nan
                 for t in range(n_frames)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(per_frame, linewidth=0.8)
    overall = np.nanmean(per_frame)
    axes[0].axhline(y=overall, color='r', linestyle='--', label=f'mean={overall:.1f}px')
    axes[0].set_xlabel('Frame'); axes[0].set_ylabel('L1 px error')
    axes[0].set_title(f'Wrist BG per-frame error')
    axes[0].legend()

    per_track = [pixel_err[vis_np[:, n], n].mean() if vis_np[:, n].any() else np.nan
                 for n in range(pixel_err.shape[1])]
    axes[1].hist([x for x in per_track if not np.isnan(x)], bins=30, color='teal')
    axes[1].set_xlabel('Mean L1 px error'); axes[1].set_title('Per-track error distribution')
    plt.suptitle(f'Wrist BG Track Error ({tracker_name})', fontsize=14)
    plt.tight_layout(); plt.show()

# ============================================================
# 4. TRACK STATIC ROBOT FK — URDF FK 2D vs tracker 2D
# ============================================================
print("\n" + "=" * 70)
print("📊 4. Track Static Robot (URDF FK vs 2D tracker)")
print("    Input: On static cameras, find robot-region tracks via PyBullet mask.")
print("           Use URDF FK to predict 2D positions at each frame t.")
print("           Compare FK-predicted 2D vs tracker-predicted 2D.")
print("=" * 70)

for cam_id in ext_cams:
    cam_data = sc_dbg['camera'][cam_id]
    if 'tracks_2d' not in cam_data:
        print(f"  ⚠️ [{cam_id}] No tracks, skipping.")
        continue

    pb_renderer.update_robot_pose(
        scene_constants['robot']['joint_positions'][0],
        gripper_state=scene_constants['robot']['gripper_positions'][0])
    urdf_tracker = URDFKinematicsTracker(pb_renderer)
    result = urdf_tracker.extract_robot_tracks(cam_id, sc_dbg, scene_state)
    traj_3d, traj_2d_fk, vis_fk, robot_indices = result

    if robot_indices is None or len(robot_indices) < 5:
        print(f"  ⚠️ [{cam_id}] <5 robot points found.")
        continue

    tracker_2d = cam_data['tracks_2d'][:, robot_indices]
    tracker_vis = cam_data['vis_2d'][:, robot_indices]
    combined_vis = vis_fk & tracker_vis
    pixel_err = np.abs(traj_2d_fk - tracker_2d).sum(axis=-1)
    valid = combined_vis & (pixel_err < 500)

    # --- 4a. VISUAL INPUT: Show robot mask + selected tracks ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    # Left: robot mask + track points at t=0
    img0 = cam_data['video_rgb'][0].copy()
    ext_t0 = scene_state[cam_id]['extrinsics'][0]
    K_np = scene_constants['camera'][cam_id]['K_mat']
    h_img, w_img = img0.shape[:2]
    # Explicitly reset robot pose in pb_renderer to t=0 before rendering mask
    pb_renderer.update_robot_pose(
        scene_constants['robot']['joint_positions'][0],
        gripper_state=scene_constants['robot']['gripper_positions'][0])
    robot_mask = pb_renderer.render_mask(ext_t0, K_np, w_img, h_img) > 0
    overlay = img0.copy()
    overlay[robot_mask] = [50, 150, 255]
    blended = cv2.addWeighted(img0, 0.6, overlay, 0.4, 0)
    axes[0].imshow(blended)
    axes[0].scatter(tracker_2d[0, :, 0], tracker_2d[0, :, 1],
                    c='lime', s=12, label=f'Robot tracks ({len(robot_indices)})', zorder=5)
    axes[0].legend(fontsize=9)
    axes[0].set_title(f'[{cam_id[:8]}] t=0: Robot Mask + Tracks', fontsize=11)
    axes[0].axis('off')

    # Mid: FK vs tracker at a mid frame
    t_show = VIS_FRAME
    img_mid = cam_data['video_rgb'][t_show].copy()
    axes[1].imshow(img_mid)
    v_t = combined_vis[t_show]
    axes[1].scatter(tracker_2d[t_show, v_t, 0], tracker_2d[t_show, v_t, 1],
                    c='lime', s=12, label='Tracker', zorder=5, alpha=0.7)
    axes[1].scatter(traj_2d_fk[t_show, v_t, 0], traj_2d_fk[t_show, v_t, 1],
                    c='red', s=12, marker='x', label='FK-predicted', zorder=5, alpha=0.7)
    for i in np.where(v_t)[0][:30]:
        axes[1].plot([tracker_2d[t_show, i, 0], traj_2d_fk[t_show, i, 0]],
                     [tracker_2d[t_show, i, 1], traj_2d_fk[t_show, i, 1]],
                     'y-', alpha=0.3, linewidth=0.5)
    err_mid = pixel_err[t_show, v_t].mean() if v_t.any() else 0
    axes[1].set_title(f'[{cam_id[:8]}] Frame {t_show}\n'
                      f'FK (red ×) vs Tracker (green ●)\nmean err={err_mid:.1f}px', fontsize=11)
    axes[1].legend(fontsize=8)
    axes[1].axis('off')

    # Right: tracking video overlay using core.visualization
    robot_tracks_video = render_2d_tracking_video(
        cam_data['video_rgb'][:min(30, n_frames)],
        traj_2d_fk[:min(30, n_frames)],
        vis_fk[:min(30, n_frames)],
        linewidth=2, tracks_leave_trace=10)
    mid = min(20, len(robot_tracks_video)-1)
    axes[2].imshow(robot_tracks_video[mid])
    axes[2].set_title(f'[{cam_id[:8]}] FK Tracks Video (frame {mid})\n'
                      f'FK-predicted robot trajectories', fontsize=11)
    axes[2].axis('off')
    plt.suptitle(f'Static Robot FK — [{cam_id[:8]}]', fontsize=14)
    plt.tight_layout()
    plt.show()

    # --- 4b. Error curves ---
    per_frame = [pixel_err[t, valid[t]].mean() if valid[t].any() else np.nan
                 for t in range(pixel_err.shape[0])]
    per_track = [pixel_err[valid[:, n], n].mean() if valid[:, n].any() else np.nan
                 for n in range(pixel_err.shape[1])]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(per_frame, linewidth=0.8)
    overall = np.nanmean(per_frame)
    axes[0].axhline(y=overall, color='r', linestyle='--', label=f'mean={overall:.1f}px')
    axes[0].set_xlabel('Frame'); axes[0].set_ylabel('L1 px error')
    axes[0].set_title(f'[{cam_id[:8]}] Robot FK per-frame error')
    axes[0].legend()

    axes[1].hist([x for x in per_track if not np.isnan(x)], bins=30, color='purple', alpha=0.7)
    axes[1].set_xlabel('Mean L1 px error')
    axes[1].set_title(f'[{cam_id[:8]}] Per-track error distribution')
    plt.suptitle(f'Static Robot FK Error ({tracker_name}) [{cam_id[:8]}]', fontsize=14)
    plt.tight_layout(); plt.show()

# ============================================================
# 5. BG OVERLAP % — visual explanation
# ============================================================
print("\n" + "=" * 70)
print("📊 5. BG Overlap % (Chamfer overlap ratio)")
print("    Computed from Chamfer distance: fraction of point pairs")
print("    within 5cm threshold. Higher = better calibration.")
print("=" * 70)

# Already computed in Chamfer section above — just show the number
metrics = evaluate_extrinsics(sc_dbg, scene_state, device,
                              pb_renderer=pb_renderer, track_anchors=anchors)
print(f"  BG Overlap: {metrics.get('bg_overlap_pct', 0):.1f}%")
print(f"  (Chamfer 5cm threshold: points closer than 5cm are 'overlapping')")

# ============================================================
# 6. SUMMARY — all metrics
# ============================================================
print("\n" + "=" * 70)
print("📊 6. Summary")
print("=" * 70)
print_metrics(metrics, f"All Metrics ({tracker_name})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Left: depth-related metrics
depth_names = ['chamfer_total', 'robot_loss_cam1', 'robot_loss_cam2', 'robot_loss_wrist']
depth_vals = [metrics.get(k, 0) for k in depth_names]
depth_labels = ['Chamfer\ntotal', 'Robot\ncam1', 'Robot\ncam2', 'Robot\nwrist']
bars = axes[0].bar(depth_labels, depth_vals, color=['#2196F3', '#FF9800', '#FF9800', '#FF9800'])
for bar, val in zip(bars, depth_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('Error (meters)'); axes[0].set_title('3D Metrics')

# Right: track metrics
track_names = ['track_reproj_wrist_bg_mean_px', 'track_reproj_wrist_bg_median_px',
               'track_reproj_static_robot_mean_px', 'track_reproj_static_robot_median_px']
track_vals = [metrics.get(k, float('nan')) for k in track_names]
track_labels = ['Wrist BG\nmean', 'Wrist BG\nmedian', 'Robot FK\nmean', 'Robot FK\nmedian']
colors = ['#4CAF50', '#81C784', '#9C27B0', '#BA68C8']
bars2 = axes[1].bar(track_labels, [v if not np.isnan(v) else 0 for v in track_vals], color=colors)
for bar, val in zip(bars2, track_vals):
    label = f'{val:.1f}' if not np.isnan(val) else 'N/A'
    axes[1].text(bar.get_x() + bar.get_width()/2, max(bar.get_height(), 0.5),
                 label, ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Error (pixels)')
axes[1].set_title(f'2D Track Metrics ({tracker_name})')
plt.suptitle(f'Extrinsics Quality Summary — {episode_id}', fontsize=14)
plt.tight_layout(); plt.show()

---
## 3. Stage 3: Tracking

Multi-view CoTracker tracking + median 3D fusion.

**Dual-Track Architecture:**
- **Track A (Environment)**: CoTracker dense 2D → 3D dedup → cross-view query → median fusion
- **Track B (Robot)**: URDF forward kinematics → per-link binding → cross-view projection

**Output format** (`tracks_3d.npz`):
- `traj_3d`: (T, N, 3) — 3D world-frame trajectories
- `vis_global`: (T, N) — global visibility mask

**Per-camera** (`{cam_id}/tracks_2d.npz`):
- `traj_2d`: (T, N, 2) — 2D pixel coordinates
- `vis_2d`: (T, N) — per-camera visibility

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 3A. 🚧 COMPUTE tracks from scratch (CoTracker + URDF + multi-view fusion)

from compute_tracks import (
    init_tracking_models,
    phase1_extract_2d_tracks, phase2_lift_and_filter,
    phase3_3d_dedup, phase4_cross_view_completion,
    phase5_median_3d_fusion, export_tracks,
)

# Init models (only first time)
if 'cotracker_model' not in dir():
    cotracker_model = init_tracking_models()

camera_ids = list(scene_constants['camera'].keys())

# Phase 1-5
scene_constants = phase1_extract_2d_tracks(cotracker_model, scene_constants, device)
per_cam_env = phase2_lift_and_filter(scene_constants, scene_state, pb_renderer)
unified_pts_3d, unified_to_cam, N_unified = phase3_3d_dedup(per_cam_env, camera_ids)
per_cam_tracks, per_cam_vis = phase4_cross_view_completion(
    cotracker_model, scene_constants, scene_state,
    per_cam_env, unified_pts_3d, unified_to_cam, N_unified, device)
(final_traj_3d, final_vis_global,
 final_per_cam_tracks, final_per_cam_vis, n_survived) = phase5_median_3d_fusion(
    scene_constants, scene_state, per_cam_tracks, per_cam_vis, N_unified)

export_tracks(scene_constants, scene_state,
              final_traj_3d, final_vis_global,
              final_per_cam_tracks, final_per_cam_vis)
print(f"✅ Stage 3 COMPUTE complete: {final_traj_3d.shape[1]} points")

In [ ]:
# @title 3B. ☁️ LOAD tracks from GCS bucket (skip Stage 3 computation)

GCS_TRACKS = "gs://dm-tapnet/mv-tap/droid/tracks"
local_tracks_cache = f"/content/droid_tracks_cache/{episode_id}"
os.makedirs(local_tracks_cache, exist_ok=True)

# Download global 3D tracks + metadata
for fname in ["tracks_3d.npz", "track_metadata.npz"]:
    gcs_path = f"{GCS_TRACKS}/{episode_id}/{fname}"
    local_path = os.path.join(local_tracks_cache, fname)
    ret = os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")
    if ret == 0:
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} not found (skipping)")

# Load global tracks
data_3d = np.load(os.path.join(local_tracks_cache, "tracks_3d.npz"))
final_traj_3d = data_3d["traj_3d"]       # (T, N, 3)
final_vis_global = data_3d["vis_global"]  # (T, N)

# Load env/robot split metadata
meta_path = os.path.join(local_tracks_cache, "track_metadata.npz")
if os.path.exists(meta_path):
    meta = np.load(meta_path)
    n_env   = int(meta["n_env"])
    n_robot = int(meta["n_robot"])
else:
    n_env, n_robot = final_traj_3d.shape[1], 0

T, N, _ = final_traj_3d.shape
print(f"  ✅ tracks_3d: {T} frames × {N} points ({n_env} env + {n_robot} robot)")

# Download per-camera 2D tracks
final_per_cam_tracks = {}
final_per_cam_vis = {}

for cam_id in scene_constants["camera"]:
    cam_cache = os.path.join(local_tracks_cache, cam_id)
    os.makedirs(cam_cache, exist_ok=True)

    gcs_cam = f"{GCS_TRACKS}/{episode_id}/{cam_id}"
    for fname in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"]:
        os.system(f"gsutil cp '{gcs_cam}/{fname}' '{cam_cache}/' > /dev/null 2>&1")

    t2d_path = os.path.join(cam_cache, "tracks_2d.npz")
    if os.path.exists(t2d_path):
        d = np.load(t2d_path)
        final_per_cam_tracks[cam_id] = d["traj_2d"]   # (T, N, 2)
        final_per_cam_vis[cam_id]    = d["vis_2d"]     # (T, N)
        print(f"  ✅ Camera [{cam_id}]: 2D tracks loaded")
    else:
        print(f"  ⚠️  Camera [{cam_id}]: tracks_2d.npz not found")

print(f"\n✅ Stage 3 LOADED from GCS — {len(final_per_cam_tracks)} cameras")

In [ ]:
# @title 3a. 📊 Track Summary Statistics
import numpy as np

T, N, _ = final_traj_3d.shape
camera_ids = list(scene_constants['camera'].keys())

# Load env/robot split if not already available
if 'n_env' not in dir() or 'n_robot' not in dir():
    try:
        meta_path = os.path.join(local_tracks_cache, "track_metadata.npz")
        meta = np.load(meta_path)
        n_env = int(meta["n_env"])
        n_robot = int(meta["n_robot"])
    except Exception:
        n_env, n_robot = N, 0

# Visibility stats
vis_per_point = final_vis_global.sum(axis=0)   # how many frames each point is visible
vis_per_frame = final_vis_global.sum(axis=1)   # how many points visible per frame

print("=" * 60)
print(f"🎯 Track Summary — Episode: {episode_id}")
print("=" * 60)
print(f"  Frames (T):         {T}")
print(f"  Total points (N):   {N}")
print(f"    ├─ Environment:   {n_env}")
print(f"    └─ Robot:         {n_robot}")
print(f"  Cameras:            {len(camera_ids)}")
print()
print(f"  Track length (frames visible per point):")
print(f"    mean:   {np.mean(vis_per_point):.1f}")
print(f"    median: {np.median(vis_per_point):.0f}")
print(f"    min:    {np.min(vis_per_point)}")
print(f"    max:    {np.max(vis_per_point)}")
print(f"    always visible: {np.mean(vis_per_point == T) * 100:.1f}%")
print()
print(f"  Points per frame:")
print(f"    mean:   {np.mean(vis_per_frame):.0f}")
print(f"    min:    {np.min(vis_per_frame)}")
print(f"    max:    {np.max(vis_per_frame)}")
print()

# 3D extent
visible_pts = final_traj_3d[final_vis_global]
bbox_min = visible_pts.min(axis=0)
bbox_max = visible_pts.max(axis=0)
bbox_size = bbox_max - bbox_min
print(f"  Scene 3D extent:")
print(f"    X: {bbox_size[0]:.3f} m")
print(f"    Y: {bbox_size[1]:.3f} m")
print(f"    Z: {bbox_size[2]:.3f} m")
print(f"    diagonal: {np.linalg.norm(bbox_size):.3f} m")
print()

# Per-camera 2D track coverage
for cam_id in camera_ids:
    if cam_id in final_per_cam_tracks:
        cam_vis = final_per_cam_vis[cam_id]
        cam_vis_t0 = cam_vis[0].sum()
        cam_avg = cam_vis.sum(axis=1).mean()
        print(f"  [{cam_id}] t=0 visible: {cam_vis_t0}/{N} "
              f"({cam_vis_t0/N*100:.1f}%) | avg/frame: {cam_avg:.0f}")

In [ ]:
# @title 3b. 🎬 Per-Camera 2D Tracking Video (multi-cam grid)
import importlib, core.visualization
importlib.reload(core.visualization)
from core.visualization import render_2d_tracking_video
import mediapy as media

camera_ids = list(scene_constants['camera'].keys())

# --- Env-only tracks (first n_env points) ---
all_frames_env = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    if cam_id not in final_per_cam_tracks:
        continue
    tracks = final_per_cam_tracks[cam_id][:, :n_env, :]
    vis = final_per_cam_vis[cam_id][:, :n_env]
    frames = render_2d_tracking_video(
        cam_data['video_rgb'], tracks, vis,
        tgt_size=(256, 456), linewidth=1, max_frames=30)
    # Add camera label
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_env.append(np.array(frames))

if all_frames_env:
    combined = np.concatenate(all_frames_env, axis=2)
    media.show_video(combined, fps=10,
                     title=f"Environment Tracks ({n_env} points) — All Cameras")

# --- Robot-only tracks (last n_robot points) ---
if n_robot > 0:
    all_frames_robot = []
    for cam_id in camera_ids:
        cam_data = scene_constants['camera'][cam_id]
        if cam_id not in final_per_cam_tracks:
            continue
        tracks = final_per_cam_tracks[cam_id][:, n_env:, :]
        vis = final_per_cam_vis[cam_id][:, n_env:]
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        for f in frames:
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        all_frames_robot.append(np.array(frames))

    if all_frames_robot:
        combined_robot = np.concatenate(all_frames_robot, axis=2)
        media.show_video(combined_robot, fps=10,
                         title=f"Robot Tracks ({n_robot} points) — All Cameras")

# --- Combined: env (rainbow) + robot (red) ---
all_frames_both = []
# Build color array: rainbow for env, red for robot
y_env = final_per_cam_tracks[camera_ids[0]][0, :n_env, 1]
norm_env = plt.Normalize(y_env.min(), y_env.max())
env_colors = (plt.cm.gist_rainbow(norm_env(y_env))[:, :3] * 255).astype(np.uint8)
robot_colors = np.full((n_robot, 3), [255, 50, 50], dtype=np.uint8)
combined_colors = np.concatenate([env_colors, robot_colors], axis=0) if n_robot > 0 else env_colors

for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    if cam_id not in final_per_cam_tracks:
        continue
    frames = render_2d_tracking_video(
        cam_data['video_rgb'],
        final_per_cam_tracks[cam_id],
        final_per_cam_vis[cam_id],
        global_colors=combined_colors,
        tgt_size=(256, 456), linewidth=1, max_frames=30)
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_both.append(np.array(frames))

if all_frames_both:
    combined_both = np.concatenate(all_frames_both, axis=2)
    media.show_video(combined_both, fps=10,
                     title=f"All Tracks ({n_env} env 🌈 + {n_robot} robot 🔴)")

In [ ]:
# @title 3c. 🌐 Animated 3D Tracks (interactive Plotly player)
import importlib, core.visualization
importlib.reload(core.visualization)
from core.visualization import show_animated_plotly_point_cloud

# Assign colors: rainbow by y-position for env, red for robot
y0 = final_traj_3d[0, :n_env, 1]
norm = plt.Normalize(y0.min(), y0.max())
env_colors_3d = (plt.cm.gist_rainbow(norm(y0))[:, :3] * 255).astype(np.uint8)
robot_colors_3d = np.full((n_robot, 3), [255, 50, 50], dtype=np.uint8)
all_colors_3d = np.concatenate([env_colors_3d, robot_colors_3d], axis=0) if n_robot > 0 else env_colors_3d

show_animated_plotly_point_cloud(
    final_traj_3d, all_colors_3d,
    title=f"3D Tracks — {n_env} env + {n_robot} robot = {N} total",
    eye_pos=(0, -0.8, -1.5))

In [ ]:
# @title 3d. 📊 Track Visibility & Quality Distributions
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Row 1: Global track stats ---

# 1a. Track length histogram (frames visible per point)
vis_per_point = final_vis_global.sum(axis=0)
axes[0, 0].hist(vis_per_point[:n_env], bins=50, color='teal', alpha=0.7, label=f'Env ({n_env})')
if n_robot > 0:
    axes[0, 0].hist(vis_per_point[n_env:], bins=50, color='red', alpha=0.5, label=f'Robot ({n_robot})')
axes[0, 0].axvline(np.mean(vis_per_point), color='k', linestyle='--',
                   label=f'mean={np.mean(vis_per_point):.1f}')
axes[0, 0].set_xlabel('Frames visible')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Track Length Distribution')
axes[0, 0].legend(fontsize=8)

# 1b. Points visible per frame over time
vis_per_frame = final_vis_global.sum(axis=1)
vis_env_per_frame = final_vis_global[:, :n_env].sum(axis=1)
axes[0, 1].plot(vis_env_per_frame, color='teal', linewidth=0.8, label='Env')
if n_robot > 0:
    vis_robot_per_frame = final_vis_global[:, n_env:].sum(axis=1)
    axes[0, 1].plot(vis_robot_per_frame, color='red', linewidth=0.8, label='Robot')
axes[0, 1].plot(vis_per_frame, color='black', linewidth=0.5, alpha=0.5, label='Total')
axes[0, 1].set_xlabel('Frame')
axes[0, 1].set_ylabel('Visible points')
axes[0, 1].set_title('Points Visible per Frame')
axes[0, 1].legend(fontsize=8)

# 1c. Per-frame 3D displacement (motion energy)
deltas = np.diff(final_traj_3d, axis=0)  # (T-1, N, 3)
delta_norms = np.linalg.norm(deltas, axis=2)  # (T-1, N)
vis_transitions = final_vis_global[:-1] & final_vis_global[1:]
mean_disp = np.array([
    np.mean(delta_norms[t, vis_transitions[t]]) * 1000 if vis_transitions[t].any() else 0
    for t in range(len(deltas))])
axes[0, 2].plot(mean_disp, color='purple', linewidth=0.8)
axes[0, 2].axhline(np.mean(mean_disp), color='r', linestyle='--',
                   label=f'mean={np.mean(mean_disp):.2f} mm')
axes[0, 2].set_xlabel('Frame')
axes[0, 2].set_ylabel('Displacement (mm)')
axes[0, 2].set_title('Mean Per-Frame 3D Displacement')
axes[0, 2].legend(fontsize=8)

# --- Row 2: Per-camera stats ---

camera_ids = list(final_per_cam_tracks.keys())

# 2a. Per-camera visibility at t=0
cam_labels = [c[:8] for c in camera_ids]
vis_t0 = [final_per_cam_vis[c][0].sum() for c in camera_ids]
bars = axes[1, 0].bar(cam_labels, vis_t0, color=['#2196F3', '#FF9800', '#4CAF50'][:len(camera_ids)])
for bar, val in zip(bars, vis_t0):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val}', ha='center', va='bottom', fontsize=9)
axes[1, 0].axhline(N, color='gray', linestyle=':', alpha=0.5, label=f'Total={N}')
axes[1, 0].set_ylabel('Visible points at t=0')
axes[1, 0].set_title('Per-Camera Coverage (t=0)')
axes[1, 0].legend(fontsize=8)

# 2b. Multi-view consistency: how many cameras see each point per frame
cam_vis_stack = np.stack([final_per_cam_vis[c] for c in camera_ids], axis=0)  # (C, T, N)
cams_per_obs = cam_vis_stack.sum(axis=0)  # (T, N)
# Histogram of cameras-per-visible-observation
cams_at_visible = cams_per_obs[final_vis_global]
axes[1, 1].hist(cams_at_visible, bins=np.arange(0.5, len(camera_ids)+1.5),
                color='steelblue', edgecolor='white', rwidth=0.8)
axes[1, 1].set_xlabel('Number of cameras')
axes[1, 1].set_ylabel('Count')
pct_multi = (cams_at_visible >= 2).sum() / len(cams_at_visible) * 100
axes[1, 1].set_title(f'Multi-View Consistency\n'
                     f'≥2 cameras: {pct_multi:.1f}%')

# 2c. Per-camera avg visible points over time
for i, cam_id in enumerate(camera_ids):
    cam_vis_t = final_per_cam_vis[cam_id].sum(axis=1)
    axes[1, 2].plot(cam_vis_t, linewidth=0.8, label=f'{cam_id[:8]}')
axes[1, 2].set_xlabel('Frame')
axes[1, 2].set_ylabel('Visible points')
axes[1, 2].set_title('Per-Camera Visibility Over Time')
axes[1, 2].legend(fontsize=8)

plt.suptitle(f'Track Quality Dashboard — {episode_id}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# @title 3e. 🎯 Reprojection Error (3D tracks → 2D per camera)
import numpy as np, matplotlib.pyplot as plt
from core.geometry import project_points_np

camera_ids = list(scene_constants['camera'].keys())
T, N, _ = final_traj_3d.shape

# For each camera, project 3D tracks to 2D and compare with stored 2D tracks
print("=" * 60)
print("🎯 Reprojection Error: project 3D tracks → 2D, compare with stored 2D")
print("=" * 60)

reproj_per_cam = {}
for cam_id in camera_ids:
    if cam_id not in final_per_cam_tracks:
        continue
    cam_data = scene_constants['camera'][cam_id]
    cam_extrinsics = scene_state[cam_id]['extrinsics']
    K = cam_data['K_mat']
    traj_2d_stored = final_per_cam_tracks[cam_id]
    vis_2d = final_per_cam_vis[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    all_errors = []
    per_frame_errors = []
    for t in range(T):
        vis_t = vis_2d[t] & final_vis_global[t]
        if not vis_t.any():
            per_frame_errors.append(np.nan)
            continue

        # Project 3D → 2D (project_points_np takes cam2world)
        pts_3d = final_traj_3d[t, vis_t]
        u_proj, v_proj, z_proj = project_points_np(pts_3d, K, cam_extrinsics[t])

        # Compare with stored 2D
        u_stored = traj_2d_stored[t, vis_t, 0]
        v_stored = traj_2d_stored[t, vis_t, 1]

        # Filter: in front of camera and in bounds
        valid = ((z_proj > 0.01) &
                 (u_proj >= 0) & (u_proj < w_img) &
                 (v_proj >= 0) & (v_proj < h_img))
        if not valid.any():
            per_frame_errors.append(np.nan)
            continue

        err = np.sqrt((u_proj[valid] - u_stored[valid])**2 +
                      (v_proj[valid] - v_stored[valid])**2)
        all_errors.append(err)
        per_frame_errors.append(np.mean(err))

    if all_errors:
        all_errs = np.concatenate(all_errors)
        reproj_per_cam[cam_id] = {
            'all_errors': all_errs,
            'per_frame': np.array(per_frame_errors),
            'mean': np.mean(all_errs),
            'median': np.median(all_errs),
            'p95': np.percentile(all_errs, 95),
        }
        print(f"  [{cam_id[:8]}] mean={np.mean(all_errs):.2f}px | "
              f"median={np.median(all_errs):.2f}px | "
              f"p95={np.percentile(all_errs, 95):.2f}px")

# --- Plot ---
if reproj_per_cam:
    n_cams = len(reproj_per_cam)
    fig, axes = plt.subplots(2, n_cams, figsize=(6 * n_cams, 8))
    if n_cams == 1:
        axes = axes.reshape(2, 1)

    for col, (cam_id, data) in enumerate(reproj_per_cam.items()):
        # Top: per-frame error curve
        axes[0, col].plot(data['per_frame'], linewidth=0.8)
        axes[0, col].axhline(data['mean'], color='r', linestyle='--',
                            label=f"mean={data['mean']:.2f}px")
        axes[0, col].set_xlabel('Frame')
        axes[0, col].set_ylabel('Reproj error (px)')
        axes[0, col].set_title(f"[{cam_id[:8]}] Per-frame")
        axes[0, col].legend(fontsize=8)

        # Bottom: error histogram
        axes[1, col].hist(data['all_errors'], bins=100, color='steelblue',
                         edgecolor='white', range=(0, min(50, data['p95'] * 2)))
        axes[1, col].axvline(data['median'], color='orange', linestyle='--',
                            label=f"median={data['median']:.2f}px")
        axes[1, col].axvline(data['p95'], color='red', linestyle='--',
                            label=f"p95={data['p95']:.2f}px")
        axes[1, col].set_xlabel('Reproj error (px)')
        axes[1, col].set_ylabel('Count')
        axes[1, col].set_title(f"[{cam_id[:8]}] Distribution")
        axes[1, col].legend(fontsize=8)

    plt.suptitle(f'Reprojection Error (3D→2D) — {episode_id}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title 3f. 🔮 3D Track Snapshot + Fused Point Cloud Overlay
import importlib, core.visualization
importlib.reload(core.visualization)
from core.visualization import show_plotly_point_cloud, render_fused_point_cloud

FRAME_IDX = 0  # @param {type:"integer"}

# --- Show fused depth point cloud at this frame (for context) ---
print(f"🌐 Fused depth point cloud at frame {FRAME_IDX} (for reference):")
render_fused_point_cloud(scene_constants, scene_state, frame_idx=FRAME_IDX,
                         max_render_points=100000, use_tint=True)

# --- Overlay 3D tracks on top ---
vis_t = final_vis_global[FRAME_IDX]
pts_t = final_traj_3d[FRAME_IDX, vis_t]

# Color: env = rainbow by y, robot = red
vis_env = vis_t[:n_env]
vis_robot = vis_t[n_env:] if n_robot > 0 else np.array([], dtype=bool)

pts_env = final_traj_3d[FRAME_IDX, :n_env][vis_env]
pts_robot = final_traj_3d[FRAME_IDX, n_env:][vis_robot] if n_robot > 0 else np.zeros((0, 3))

y_e = pts_env[:, 1] if len(pts_env) > 0 else np.array([0])
norm_e = plt.Normalize(y_e.min(), y_e.max())
colors_env = (plt.cm.gist_rainbow(norm_e(y_e))[:, :3] * 255).astype(np.uint8)
colors_robot = np.full((len(pts_robot), 3), [255, 50, 50], dtype=np.uint8)

all_pts = np.concatenate([pts_env, pts_robot], axis=0)
all_colors = np.concatenate([colors_env, colors_robot], axis=0)

print(f"\n🎯 3D tracks at frame {FRAME_IDX}: {len(pts_env)} env + {len(pts_robot)} robot")
show_plotly_point_cloud(
    all_pts, all_colors,
    title=f"3D Tracks at Frame {FRAME_IDX} — {len(all_pts)} points",
    max_points=50000, eye_pos=(0, -0.8, -1.5))

---
## Summary

```
per stage:
  A. 🚧 COMPUTE — run from scratch (debug single episode)
  B. ☁️ LOAD    — pull from GCS   (skip to later stages)

droid/
├── compute_depth.py          → Stage 1
├── compute_extrinsics.py     → Stage 2
├── compute_tracks.py         → Stage 3
├── core/                     → Shared modules
│   ├── geometry.py, io.py, depth.py, physics.py, tracking.py, visualization.py
```

Typical debug workflow:
1. Run Stage 1 once → `run_parallel.sh` on GCP → results on GCS
2. Open notebook → **1B. Load** depth from GCS → 🚧 debug Stage 2
3. Stage 2 works → `run_parallel.sh --stage 2` → results on GCS
4. Open notebook → **1B + 2B. Load** both → 🚧 debug Stage 3